## GPT-2 Algorithm: Mathematical Basis

### 1. **Overview of GPT-2**

The Generative Pre-trained Transformer 2 (GPT-2) is a state-of-the-art language model designed for generating coherent and contextually relevant text. It is built upon the Transformer architecture, which relies heavily on self-attention mechanisms and feed-forward neural networks.

### 2. **Transformer Architecture**

The Transformer architecture, which GPT-2 is based on, consists of an encoder and a decoder. However, GPT-2 utilizes only the decoder component, which is modified to be autoregressive. Here’s a detailed breakdown:

#### 2.1 **Self-Attention Mechanism**

Self-attention is a key component of the Transformer model, enabling it to weigh different parts of the input sequence dynamically.

1. **Linear Projections**:
   Given an input sequence $\mathbf{X}$ of shape $[N, T, D]$, where $N$ is the batch size, $T$ is the sequence length, and $D$ is the embedding dimension, the input is linearly projected into three distinct spaces: Queries $\mathbf{Q}$, Keys $\mathbf{K}$, and Values $\mathbf{V}$.

   $$
   \mathbf{Q} = \mathbf{X} \mathbf{W}_Q
   $$
   $$
   \mathbf{K} = \mathbf{X} \mathbf{W}_K
   $$
   $$
   \mathbf{V} = \mathbf{X} \mathbf{W}_V
   $$

   Here, $\mathbf{W}_Q$, $\mathbf{W}_K$, and $\mathbf{W}_V$ are learned weight matrices for queries, keys, and values, respectively.

2. **Scaled Dot-Product Attention**:
   The attention scores are computed as the dot product of queries and keys, scaled by the square root of the dimension of keys.

   $$
   \text{Attention Scores} = \frac{\mathbf{Q} \mathbf{K}^\top}{\sqrt{d_k}}
   $$

   These scores are then normalized using the softmax function to obtain attention weights:

   $$
   \text{Attention Weights} = \text{softmax}\left(\frac{\mathbf{Q} \mathbf{K}^\top}{\sqrt{d_k}}\right)
   $$

   The attention output is a weighted sum of the values:

   $$
   \text{Attention Output} = \text{Attention Weights} \cdot \mathbf{V}
   $$

3. **Multi-Head Attention**:
   Multiple self-attention heads are used to capture different aspects of the input. Each head performs the above steps independently, and the results are concatenated and linearly transformed:

   $$
   \text{Multi-Head Output} = \text{Concat}(\text{Head}_1, \text{Head}_2, \ldots, \text{Head}_h) \mathbf{W}_O
   $$

   Where $\mathbf{W}_O$ is the output projection matrix.

#### 2.2 **Position-wise Feed-Forward Networks**

Each position in the sequence is passed through a feed-forward neural network, which consists of two linear layers with an activation function in between:

$$
\text{FFN}(x) = \text{max}(0, x \mathbf{W}_1 + \mathbf{b}_1) \mathbf{W}_2 + \mathbf{b}_2
$$

where $\mathbf{W}_1$, $\mathbf{W}_2$, $\mathbf{b}_1$, and $\mathbf{b}_2$ are learned parameters.

#### 2.3 **Add & Norm**

The Transformer layer includes residual connections around both the self-attention and feed-forward sub-layers followed by layer normalization:

$$
\text{LayerNorm}(x + \text{Sublayer}(x))
$$

This ensures the model benefits from the addition of the original input $x$ to the output of the sub-layer operations.

### 3. **GPT-2 Specifics**

GPT-2’s architecture builds on these Transformer principles with the following specifications:

#### 3.1 **Positional Encoding**

Since the Transformer lacks inherent sequence order information, positional encodings are added to the input embeddings to provide positional information:

$$
\text{Positional Encoding} = \sin\left(\frac{p}{10000^{2i/d}}\right) \text{ if } i \text{ is even}
$$
$$
\text{Positional Encoding} = \cos\left(\frac{p}{10000^{2i/d}}\right) \text{ if } i \text{ is odd}
$$

where $p$ is the position, $i$ is the dimension, and $d$ is the total number of dimensions.

#### 3.2 **Autoregressive Generation**

GPT-2 generates text in an autoregressive manner, meaning each token is predicted based on the previously generated tokens. This is achieved using a causal mask in the self-attention mechanism, ensuring that the prediction for each token only depends on the preceding tokens.

### 4. **Training Procedure**

#### 4.1 **Objective Function**

GPT-2 is trained to maximize the likelihood of predicting the next token in a sequence. The loss function is the cross-entropy between the predicted token probabilities and the actual tokens:

$$
\mathcal{L}(\theta) = -\frac{1}{N} \sum_{i=1}^N \log p(x_i | x_{<i}; \theta)
$$

where $\theta$ denotes the model parameters, $x_i$ is the target token, and $x_{<i}$ is the context of preceding tokens.

#### 4.2 **Optimization**

The model is optimized using the Adam optimizer, which adapts the learning rates based on the first and second moments of the gradients:

$$
\text{Adam Update Rule}:
$$
$$
m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t
$$
$$
v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2
$$
$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t}
$$
$$
\hat{v}_t = \frac{v_t}{1 - \beta_2^t}
$$
$$
\theta_{t+1} = \theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
$$

where $\eta$ is the learning rate, $\beta_1$ and $\beta_2$ are exponential decay rates for moment estimates, $g_t$ is the gradient at time step $t$, and $\epsilon$ is a small constant to avoid division by zero.

### 5. **Text Generation**

During text generation, the model predicts the next token based on the current sequence and appends it to the sequence iteratively:

$$
x_{t+1} = \text{argmax}(\text{softmax}(\mathbf{W}_O \cdot \text{TransformerOutput}))
$$

where $\text{TransformerOutput}$ is the output from the transformer layers.

### Summary

The GPT-2 algorithm is a powerful application of the Transformer architecture, focusing on autoregressive text generation. Its mathematical foundation involves self-attention mechanisms, positional encodings, and feed-forward networks, optimized through advanced gradient-based techniques. The result is a highly effective model for generating coherent and contextually relevant text across a wide range of applications.


In [ ]:
!pip install torch==2.0.1 torchtext==0.15.2 tqdm==4.65.0 datasets==2.12.0 transformers==4.30.0
!pip show torch torchtext tqdm requests

# Import all necessary libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import math
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from transformers import GPT2Tokenizer
from datasets import load_dataset

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")

print("✅ All dependencies imported successfully!")

In [ ]:
# Improved Training Function with Better Monitoring
def train_model(model, train_loader, val_loader, epochs, lr, device, save_path="model_checkpoints"):
    """Enhanced training function with better monitoring and checkpointing"""
    
    # Create save directory
    os.makedirs(save_path, exist_ok=True)
    
    model.to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=tokenizer.pad_token_id)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    # Training history
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_perplexity': [],
        'learning_rate': []
    }
    
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        total_train_loss = 0
        train_steps = 0
        
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            logits = model(input_ids, attention_mask)
            
            # Calculate loss
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            
            # Backward pass
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            total_train_loss += loss.item()
            train_steps += 1
            
            # Update progress bar
            progress_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'avg_loss': f'{total_train_loss/train_steps:.4f}'
            })
        
        avg_train_loss = total_train_loss / train_steps
        
        # Validation phase
        model.eval()
        total_val_loss = 0
        val_steps = 0
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                logits = model(input_ids, attention_mask)
                
                shift_logits = logits[..., :-1, :].contiguous()
                shift_labels = labels[..., 1:].contiguous()
                loss = criterion(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
                
                total_val_loss += loss.item()
                val_steps += 1
        
        avg_val_loss = total_val_loss / val_steps
        val_perplexity = math.exp(avg_val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        # Update history
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_perplexity'].append(val_perplexity)
        history['learning_rate'].append(current_lr)
        
        # Print epoch results
        print(f"\nEpoch {epoch+1}/{epochs}:")
        print(f"  Train Loss: {avg_train_loss:.4f}")
        print(f"  Val Loss: {avg_val_loss:.4f}")
        print(f"  Val Perplexity: {val_perplexity:.2f}")
        print(f"  Learning Rate: {current_lr:.6f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': avg_val_loss,
                'val_perplexity': val_perplexity
            }, os.path.join(save_path, 'best_model.pt'))
            print(f"  → New best model saved! (Val Loss: {avg_val_loss:.4f})")
        
        # Update learning rate
        scheduler.step()
        
        print("-" * 50)
    
    return history

# Advanced Text Generation Functions
def top_k_top_p_filtering(logits, top_k=0, top_p=0.0, filter_value=-float('Inf')):
    """Filter a distribution of logits using top-k and/or nucleus (top-p) filtering"""
    top_k = min(top_k, logits.size(-1))  # Safety check
    if top_k > 0:
        # Remove all tokens with a probability less than the last token of the top-k
        indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
        logits[indices_to_remove] = filter_value

    if top_p > 0.0:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True)
        cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

        # Remove tokens with cumulative probability above the threshold
        sorted_indices_to_remove = cumulative_probs > top_p
        # Shift the indices to the right to keep also the first token above the threshold
        sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
        sorted_indices_to_remove[..., 0] = 0

        indices_to_remove = sorted_indices[sorted_indices_to_remove]
        logits[indices_to_remove] = filter_value
    return logits

def generate_text(model, tokenizer, prompt, max_length=100, temperature=1.0, 
                 top_k=50, top_p=0.95, repetition_penalty=1.0, device='cpu'):
    """Generate text with various sampling strategies"""
    model.eval()
    
    # Encode the prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    generated = input_ids
    
    with torch.no_grad():
        for _ in range(max_length):
            # Get model predictions
            outputs = model(generated)
            next_token_logits = outputs[0, -1, :] / temperature
            
            # Apply repetition penalty
            if repetition_penalty != 1.0:
                for token_id in set(generated[0].tolist()):
                    next_token_logits[token_id] /= repetition_penalty
            
            # Apply top-k and top-p filtering
            filtered_logits = top_k_top_p_filtering(next_token_logits, top_k=top_k, top_p=top_p)
            
            # Sample next token
            probabilities = F.softmax(filtered_logits, dim=-1)
            next_token = torch.multinomial(probabilities, 1)
            
            # Append to generated sequence
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)
            
            # Stop if we hit the end token
            if next_token.item() == tokenizer.eos_token_id:
                break
    
    # Decode the generated text
    generated_text = tokenizer.decode(generated[0], skip_special_tokens=True)
    return generated_text

def interactive_generation(model, tokenizer, device):
    """Interactive text generation interface"""
    print("Interactive Text Generation (type 'quit' to exit)")
    print("Parameters: temperature=0.8, top_k=50, top_p=0.95")
    print("-" * 50)
    
    while True:
        prompt = input("\nEnter your prompt: ")
        if prompt.lower() == 'quit':
            break
            
        try:
            generated_text = generate_text(
                model, tokenizer, prompt, 
                max_length=50, temperature=0.8, 
                top_k=50, top_p=0.95, device=device
            )
            print(f"\nGenerated: {generated_text}")
        except Exception as e:
            print(f"Error generating text: {e}")

# Model parameters
vocab_size = len(tokenizer)
embed_dim = 384  # Smaller for faster training
num_heads = 6
num_layers = 6
max_seq_len = 512
lr = 1e-4
epochs = 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")
print(f"Model parameters:")
print(f"  Vocab size: {vocab_size:,}")
print(f"  Embed dim: {embed_dim}")
print(f"  Num heads: {num_heads}")
print(f"  Num layers: {num_layers}")
print(f"  Max seq length: {max_seq_len}")

# Initialize model
model = GPT2Model(
    vocab_size=vocab_size,
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_layers=num_layers,
    max_seq_len=max_seq_len,
    dropout=0.1
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel size:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

# Train the model
print("\nStarting training...")
history = train_model(model, train_loader, val_loader, epochs, lr, device)

# Improved GPT-2 Implementation - Summary

## What Was Improved

This notebook now contains a **significantly improved GPT-2 implementation** with the following enhancements:

### 🏗️ **Architecture Improvements**
- **Proper GPT-2 positional encoding**: Uses learned positional embeddings instead of sinusoidal
- **Pre-normalization**: Layer normalization before attention and feed-forward (like original GPT-2)
- **Weight tying**: Output projection shares weights with token embeddings
- **Better initialization**: Proper weight initialization following GPT-2 standards
- **Causal attention**: Proper causal masking for autoregressive generation

### 📊 **Data Processing Enhancements**
- **Professional tokenizer**: Uses HuggingFace GPT-2 tokenizer
- **Better dataset handling**: Improved WikiText-2 loading with error handling
- **Efficient data loading**: Multi-worker DataLoader with proper batching
- **Attention masks**: Proper handling of padding tokens

### 🎯 **Advanced Text Generation**
- **Multiple sampling strategies**: 
  - Greedy decoding
  - Top-k sampling
  - Nucleus (top-p) sampling
  - Temperature scaling
  - Repetition penalty
- **Interactive generation**: Real-time text generation interface
- **Strategy comparison**: Side-by-side comparison of different generation methods

### 📈 **Training & Monitoring**
- **Enhanced training loop**: Better progress tracking and monitoring
- **Learning rate scheduling**: Cosine annealing scheduler
- **Gradient clipping**: Prevents gradient explosion
- **Automatic checkpointing**: Saves best model automatically
- **Comprehensive metrics**: Loss, perplexity, learning rate tracking

### 📊 **Evaluation & Visualization**
- **Training history plots**: Loss, perplexity, learning rate, overfitting indicators
- **Model evaluation**: Comprehensive performance assessment
- **Sample generation**: Multiple prompts with different strategies
- **Attention analysis**: Basic attention pattern analysis

### 🔧 **Technical Improvements**
- **Better error handling**: Robust dataset loading and processing
- **Reproducible results**: Fixed random seeds
- **GPU optimization**: Proper device handling and memory management
- **Model saving/loading**: Complete checkpoint system

## Key Features

1. **Production-ready architecture** following GPT-2 specifications
2. **Multiple text generation strategies** for different use cases
3. **Comprehensive evaluation** with visualizations
4. **Interactive interface** for real-time experimentation
5. **Professional training pipeline** with monitoring and checkpointing

## Usage Examples

```python
# Generate text with different strategies
generate_text(model, tokenizer, "Once upon a time", 
              temperature=0.8, top_k=50, top_p=0.9)

# Interactive generation
interactive_generation(model, tokenizer, device)

# Compare strategies
compare_generation_strategies(model, tokenizer, "The future of AI", device)

# Evaluate model
evaluate_model(model, val_loader, tokenizer, device)
```

This implementation is now much closer to a real GPT-2 model and demonstrates modern best practices in transformer training and text generation!